# 02 — DL com Sinal Bruto (raw)

Baseline DL sobre o EEG cru (6 canais), sem transformação wavelet.

**Recomendado:** rodar tudo via fila GPU:
```bash
cd tests/scp1 && MODES=raw python run_dl_queue.py --fresh
```
Abaixo, um exemplo **inline** (1 modelo).

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    SCP1_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist, N_JOBS_QUARTER,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', SCP1_CONFIG['dataset_name'], '| classes:', SCP1_CONFIG['n_classes'],
      '| seq_len:', SCP1_CONFIG['sequence_length'], '| canais:', SCP1_CONFIG['n_features'])

In [ ]:
from src.pipeline_queue import SCP1ExperimentPipeline
import experiment_config as C
MODE = 'raw'
MODEL = 'CNN'
if MODE in ('raw', 'db4'):
    cfg = {**C.DL_MODELS_CONFIG[MODEL], **C.DL_TRAINING_CONFIG}
    if MODE == 'db4':
        cfg.update({k: C.LEARNED_WAVELET_CONFIG[k] for k in ('levels','align')})
else:
    cfg = {**C.LEARNED_WAVELET_MODELS_CONFIG[MODEL], **C.DL_TRAINING_CONFIG, **C.LEARNED_WAVELET_CONFIG}
cfg['epochs'] = 20
pipe = SCP1ExperimentPipeline(MODEL, MODE, cfg, 0, str(RESULTS_DIR / '_notebook_demo'), data_dir=str(DATA_DIR))
metrics = pipe.run()
print({k: round(v,4) for k,v in metrics.items() if k.startswith('test_')})